### Step 1: Import libraries and API calls

In [57]:
import os
from openai import OpenAI
from litellm import completion
import json 

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("Api key not available")
else:
    print("Api key up and ready!")

Api key up and ready!


### Step 2: Set up pushover

In [58]:
pushover_user = os.getenv("PUSHOVER_USER_KEY")
pushover_token = os.getenv("PUSHOVER_TOKEN_KEY")
pushover_url = "https://api.pushover.net/1/messages.json"

In [59]:
import requests

def send_notification(message:str):
    payload  = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload)
    return response
    # print(f"Pushover response: {response.status_code} {response.text}")

### Step 3: Describe pushover as an LLM tool

In [60]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends push notification to the phone via pushover. Use this to alert the use about important events completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties":{
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add pushover to the list of tools for LLM

In [61]:
tools = [{"type": "function", "function": send_notification_function}]

### Create a new function, describe it, and add it too the list of tools

In [62]:
import random

# Simulate rolling a six sided die
def roll_dice():
    result = random.randint(1,6)
    return result

# Describe the function
roll_dice_function = {
    "name": "roll_dice",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, decisions or random number generation.",
    "parameters": {
        "type": "object",
        "properties":{},
        "required": []
    }

}

# Add function (append modifies the list in place and returns None, so don't reassign)
tools.append({"type": "function", "function": roll_dice_function})

### Step 5: Calling the tool from our LLM

In [63]:
def handle_tool_call(tool_calls):
    tool_result = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        # print(f"Calling function {function_name}")
        
        # Route to the appropriate function based on the function name
        if function_name == "send_notification":
            # Actually send the notification
            send_notification(args["message"])
            content = f"Notification sent {args['message']}"
        elif function_name == "roll_dice":
            content = f"Rolled: {roll_dice()}"
        
        else:
            content = f"Unknown function {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }

        tool_result.append(tool_call_result)

    return tool_result

In [64]:
client = OpenAI()

messages=[
        {"role":"user", "content": "Please do two things \
         1. I'd like to roll 2 dice and send me a notification with the results and \
        2. Send me a separate notification saying Hello"          
        }
    ]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)
message = response.choices[0].message

# Check if model wants to call a tool 
while message.tool_calls:
    # handle tool call
    tool_result = handle_tool_call(message.tool_calls) # Whole list of tool calls on purpose
    # Add messagge to context ie messages 
    messages.append(message)
    # Add info about tool call to "context" messages
    messages.extend(tool_result)  # changed from append since we are combining two lists
    # Invoke llm one more time to get its updated response
    response = client.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        tools=tools   # will be added in the future
    )
    message = response.choices[0].message

    # Consider adding protection from infinite consecutive tool calling.
print(message.content)


I rolled two dice for you with the results 3 and 4. I've sent you a notification with these results. Additionally, I've sent you a separate notification saying "Hello."
